# RQ3 · The human vocabulary

When a path *is* human-readable, **what words are used**, and how does that
vocabulary change across 2004–2024? Complements RQ1's rates with the actual
content of the readable web.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # repo root (this notebook lives in analysis/)
sys.path.insert(0, os.path.abspath('.'))
import config, readability as rb, eot_segments as es
import duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_rows', 120); pd.set_option('display.width', 200)
YEAR_ORDER = ['2004','2008','2012','2016','2020','2024']
CLS_COLORS = {'human':'#2c7fb8','acronym':'#fec44f','machine':'#de2d26'}

DBS = config.discover_domain_dbs('cdxj')
print(f"{len(DBS)}/15 domain DBs found:", ", ".join(DBS) or "(none — run on the server)")

In [ ]:
seg = es.load_segments(DBS)
human = seg[seg.cls=='human'].copy()
# tokenize human segments into words (hyphen/underscore/etc. -> words), keep real words
import re
def words(s):
    return [t for t in re.split(r'[^a-z]+', s.lower()) if len(t)>=3 and rb.word_score(t) >= rb.WORD_ZIPF_MIN]
human['words'] = human['seg'].map(words)
wexp = human.explode('words').dropna(subset=['words']).rename(columns={'words':'word'})
print(f"{wexp.word.nunique():,} distinct human words")

## 1. Top human words overall (type-level: distinct segments)

Each distinct segment votes once, so this is the design vocabulary, not traffic.

In [ ]:
typ = wexp.drop_duplicates(['domain','crawl_year','pos','seg','word'])
top = typ['word'].value_counts().head(30)
plt.figure(figsize=(9,8))
top.iloc[::-1].plot(kind='barh', color='#2c7fb8')
plt.title('Top 30 human path words (distinct segments)'); plt.xlabel('distinct segments using word')
plt.tight_layout(); plt.savefig('rq3_top_words.png', dpi=150, bbox_inches='tight'); plt.show()

## 2. Vocabulary shift — top words across years (heatmap)

For the overall top-20 words, share of that year's human segments containing
them. Reveals which concepts rose (e.g. `data`, `news`) or faded (e.g. `cfm`,
`gov`, legacy words).

In [ ]:
top20 = top.head(20).index.tolist()
yr_tot = typ.groupby('crawl_year')['seg'].nunique()
sub = typ[typ.word.isin(top20)]
cnt = sub.groupby(['word','crawl_year'])['seg'].nunique().unstack('crawl_year').fillna(0)
share = (100*cnt.div(yr_tot, axis=1)).round(1)
share = share[[y for y in YEAR_ORDER if y in share.columns]].reindex(top20)
plt.figure(figsize=(9,8))
sns.heatmap(share, annot=True, fmt='.1f', cmap='mako',
            cbar_kws={'label':'% of year\'s human segments'}, linewidths=.5)
plt.title('Top-20 human words: share of readable vocabulary by year')
plt.xlabel('crawl year'); plt.ylabel(''); plt.tight_layout()
plt.savefig('rq3_word_shift.png', dpi=150, bbox_inches='tight'); plt.show()

## 3. Rising vs fading words (earliest vs latest year present)

Largest gainers and losers in vocabulary share between the first and last
crawl years available.

In [ ]:
years_present = [y for y in YEAR_ORDER if y in share.columns]
first, last = years_present[0], years_present[-1]
allw = typ[typ.word.isin(top.head(80).index)]
cnt2 = allw.groupby(['word','crawl_year'])['seg'].nunique().unstack('crawl_year').fillna(0)
sh2 = 100*cnt2.div(yr_tot, axis=1)
delta = (sh2[last] - sh2[first]).sort_values()
fig, (a1,a2) = plt.subplots(1,2, figsize=(14,6))
delta.head(15).plot(kind='barh', ax=a1, color='#de2d26'); a1.set_title(f'Fading ({first}→{last})')
delta.tail(15).plot(kind='barh', ax=a2, color='#2c7fb8'); a2.set_title(f'Rising ({first}→{last})')
for a in (a1,a2): a.set_xlabel('Δ share (pct points)')
plt.tight_layout(); plt.savefig('rq3_rising_fading.png', dpi=150, bbox_inches='tight'); plt.show()
typ.to_csv('rq3_human_words.csv', index=False); print('saved rq3_human_words.csv')